In [0]:
# ==========================================================
# SILVER curation - vehicle production (incremental from bronze)
# ==========================================================
# Fetches ONLY the bronze rows with ingest_ts > this table's own
# watermark (control_metadata row owned by this script), cleans and
# dedups them, appends to silver, then advances its own watermark to
# the max ingest_ts consumed. Bronze writes nothing to the control
# table, and each silver table owns only its own row, so layers never
# fight over the same watermark.

from datetime import datetime
import uuid

from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_TABLE = "automotive_catalog.automotive_mfg.bronze_vehicle_production"
SILVER_TABLE = "automotive_catalog.automotive_mfg.silver_vehicle_production"

EPOCH = datetime(1900, 1, 1)


def audit_log(message):
    print(f"[AUDIT] {message}")


def generate_batch_id():
    return str(uuid.uuid4())


audit_log("Starting Silver Vehicle Processing")

# 1) Read this table's own watermark
wm_row = spark.sql(f"""
    SELECT last_processed_ts
    FROM automotive_catalog.automotive_mfg.control_metadata
    WHERE table_name = 'silver_vehicle_production'
""").collect()
last_wm = wm_row[0]["last_processed_ts"] if wm_row else None
if last_wm is None:
    last_wm = EPOCH
audit_log(f"Incremental fetch: bronze rows with ingest_ts > {last_wm}")

# 2) Incremental fetch from bronze
inc_df = spark.table(BRONZE_TABLE).filter(F.col("ingest_ts") > F.lit(last_wm))
row_count = inc_df.count()

if row_count == 0:
    audit_log("No new bronze rows - nothing to process")
else:
    # High-watermark of this increment (advance only after a
    # successful write below)
    new_wm = inc_df.agg(F.max("ingest_ts")).collect()[0][0]
    batch_id = generate_batch_id()

    clean_df = (
        inc_df
        .filter(F.col("vin_number").isNotNull())
        .filter(F.col("plant_code").isNotNull())
        .withColumn("production_status", F.upper(F.trim(F.col("production_status"))))
        .withColumn("production_stage", F.upper(F.trim(F.col("production_stage"))))
        .withColumn("model_name", F.upper(F.trim(F.col("model_name"))))
    )

    window_spec = Window.partitionBy("vin_number").orderBy(F.col("ingest_ts").desc())

    dedup_df = (
        clean_df
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn", "ingest_ts", "source_file_name", "load_date")
        .withColumn("processed_ts", F.current_timestamp())
        .withColumn("batch_run_id", F.lit(batch_id))
    )

    written_count = dedup_df.count()
    dedup_df.write.format("delta").mode("append").saveAsTable(SILVER_TABLE)

    # 3) Advance own watermark only after a successful write
    spark.sql(f"""
        UPDATE automotive_catalog.automotive_mfg.control_metadata
        SET last_processed_ts = '{new_wm}', updated_at = current_timestamp()
        WHERE table_name = 'silver_vehicle_production'
    """)

    audit_log(
        f"Read {row_count} bronze rows, wrote {written_count} silver rows, "
        f"new watermark {new_wm}"
    )

audit_log("Completed Silver Vehicle Processing")

In [0]:
%sql
SELECT * FROM automotive_catalog.automotive_mfg.silver_vehicle_production